# 模型保存与加载 — PyTorch 版

> 本笔记是 [TensorFlow/Keras 版本](05-保存和加载模型.ipynb) 的 PyTorch 等价实现。

本教程介绍 PyTorch 模型的保存和加载方法，这是模型部署和继续训练的关键技能。

## 学习目标

1. 掌握 `torch.save` / `torch.load` 保存与加载模型
2. 理解 `state_dict` 的概念及使用
3. 了解保存完整模型 vs 仅保存权重的区别
4. 在训练循环中实现 Checkpoint 与 EarlyStopping
5. 保存优化器状态以恢复训练
6. 使用 ONNX 导出模型用于部署

## 1. 环境配置与模型准备

In [ ]:
import os

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# 设置随机种子 / Set random seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# 选择设备 / Select device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"PyTorch版本: {torch.__version__}")
print(f"使用设备: {device}")

In [ ]:
# 准备数据 / Prepare data
housing = fetch_california_housing()
X_train_full, X_test, y_train_full, y_test = train_test_split(
    housing.data, housing.target, test_size=0.2, random_state=RANDOM_SEED
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=RANDOM_SEED
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

# 转为 PyTorch 张量 / Convert to PyTorch tensors
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train).unsqueeze(1)
X_valid_t = torch.FloatTensor(X_valid)
y_valid_t = torch.FloatTensor(y_valid).unsqueeze(1)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test).unsqueeze(1)

# 创建 DataLoader / Create DataLoaders
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

valid_dataset = TensorDataset(X_valid_t, y_valid_t)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)

test_dataset = TensorDataset(X_test_t, y_test_t)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"训练集: {X_train_t.shape}")
print(f"验证集: {X_valid_t.shape}")
print(f"测试集: {X_test_t.shape}")

In [ ]:
# 构建模型 / Build model
class MLP(nn.Module):
    """简单的多层感知机回归模型
    Simple MLP regression model.
    """
    def __init__(self, input_dim=8, hidden_dim=30):
        super().__init__()
        self.hidden1 = nn.Linear(input_dim, hidden_dim)
        self.hidden2 = nn.Linear(hidden_dim, hidden_dim)
        self.output = nn.Linear(hidden_dim, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.hidden1(x))
        x = self.relu(self.hidden2(x))
        return self.output(x)


def create_model():
    """创建模型并移到设备上
    Create model and move to device.
    """
    model = MLP(input_dim=8, hidden_dim=30).to(device)
    return model


model = create_model()
print(model)
print(f"\n参数数量: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 训练辅助函数 / Training helper functions
def train_one_epoch(model, loader, criterion, optimizer, device):
    """训练一个 epoch / Train one epoch."""
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
    return total_loss / len(loader.dataset)


def evaluate(model, loader, criterion, device):
    """评估模型 / Evaluate model."""
    model.eval()
    total_loss = 0.0
    total_mae = 0.0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            total_loss += criterion(y_pred, y_batch).item() * X_batch.size(0)
            total_mae += torch.abs(y_pred - y_batch).sum().item()
    n = len(loader.dataset)
    return total_loss / n, total_mae / n


def train_model(model, train_loader, valid_loader, epochs=20, lr=1e-3, device=device):
    """完整训练流程 / Full training pipeline."""
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = {'train_loss': [], 'val_loss': [], 'val_mae': []}

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_mae = evaluate(model, valid_loader, criterion, device)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_mae'].append(val_mae)
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d} | train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f} | val_mae: {val_mae:.4f}")

    return model, optimizer, history


# 训练模型 / Train model
model, optimizer, history = train_model(model, train_loader, valid_loader, epochs=20)

# 评估原始模型 / Evaluate original model
criterion = nn.MSELoss()
original_loss, original_mae = evaluate(model, test_loader, criterion, device)
print(f"\n原始模型 - MSE: {original_loss:.4f}, MAE: {original_mae:.4f}")

## 2. 保存和加载模型权重 (`state_dict`)

PyTorch 中最推荐的保存方式是仅保存模型的 `state_dict`（即参数字典）。
这相当于 Keras 中的 `model.save_weights()`。

### `state_dict` 是什么？

`state_dict` 是一个 Python 字典，将每一层的参数张量映射到该层名称。
例如 `hidden1.weight`、`hidden1.bias` 等。

In [ ]:
# 查看 state_dict 的内容 / Inspect state_dict
print("模型的 state_dict 键:")
for key, tensor in model.state_dict().items():
    print(f"  {key}: {tensor.shape}")

In [ ]:
# 保存 state_dict / Save state_dict
os.makedirs('saved_models', exist_ok=True)
weights_path = 'saved_models/my_model_weights.pth'

torch.save(model.state_dict(), weights_path)
print(f"权重已保存到: {weights_path}")
print(f"文件大小: {os.path.getsize(weights_path) / 1024:.2f} KB")

In [ ]:
# 加载 state_dict / Load state_dict
# 必须先创建相同架构的模型，再加载权重
# Must create a model with the same architecture first, then load weights
new_model = create_model()

# 加载权重前的表现 / Performance before loading weights
before_loss, before_mae = evaluate(new_model, test_loader, criterion, device)
print(f"加载权重前 - MSE: {before_loss:.4f}, MAE: {before_mae:.4f}")

# 加载权重 / Load weights
new_model.load_state_dict(torch.load(weights_path, weights_only=True))
new_model.to(device)

# 加载权重后的表现 / Performance after loading weights
after_loss, after_mae = evaluate(new_model, test_loader, criterion, device)
print(f"加载权重后 - MSE: {after_loss:.4f}, MAE: {after_mae:.4f}")
print(f"与原始模型一致: {np.isclose(original_loss, after_loss)}")

## 3. 保存完整模型（含架构）

### 方法一：保存整个模型对象（不推荐）

使用 `pickle` 保存整个模型对象，依赖于源码路径，跨环境可移植性差。

### 方法二：保存 state_dict + 模型类定义（推荐）

将模型类定义保留在代码中，只保存和加载 `state_dict`。这是 PyTorch 官方推荐的方式。

### 方法三：保存 state_dict + 类信息到 checkpoint（灵活）

将类名等信息与 `state_dict` 一起保存，便于自动重建模型。

In [ ]:
# 方法一：保存整个模型对象 / Method 1: Save entire model object (NOT recommended)
full_model_path = 'saved_models/my_full_model.pt'
torch.save(model, full_model_path)

print(f"完整模型已保存到: {full_model_path}")
print(f"文件大小: {os.path.getsize(full_model_path) / 1024:.2f} KB")
print("\n⚠️ 注意: 这种方式使用 pickle 序列化，跨环境可移植性差，不推荐用于生产环境")

In [ ]:
# 加载完整模型对象（需要模型类在当前作用域中可用）
# Load entire model object (model class must be available in current scope)
loaded_full_model = torch.load(full_model_path, weights_only=False)
loaded_full_model.to(device)

full_loss, full_mae = evaluate(loaded_full_model, test_loader, criterion, device)
print(f"加载完整模型 - MSE: {full_loss:.4f}, MAE: {full_mae:.4f}")
print(f"与原始模型一致: {np.isclose(original_loss, full_loss)}")

In [ ]:
# 方法二：保存 state_dict + 类信息 / Method 2: Save state_dict with class metadata
checkpoint_meta_path = 'saved_models/my_model_checkpoint.pth'

checkpoint = {
    'model_state_dict': model.state_dict(),
    'model_class': model.__class__.__name__,
    'input_dim': 8,
    'hidden_dim': 30,
}
torch.save(checkpoint, checkpoint_meta_path)

print(f"Checkpoint 已保存到: {checkpoint_meta_path}")
print(f"文件大小: {os.path.getsize(checkpoint_meta_path) / 1024:.2f} KB")

In [ ]:
# 从 checkpoint 加载 / Load from checkpoint
loaded_ckpt = torch.load(checkpoint_meta_path, weights_only=False)

# 根据保存的类信息重建模型 / Rebuild model from saved metadata
if loaded_ckpt['model_class'] == 'MLP':
    restored_model = MLP(
        input_dim=loaded_ckpt['input_dim'],
        hidden_dim=loaded_ckpt['hidden_dim']
    ).to(device)

restored_model.load_state_dict(loaded_ckpt['model_state_dict'])

ckpt_loss, ckpt_mae = evaluate(restored_model, test_loader, criterion, device)
print(f"Checkpoint 加载模型 - MSE: {ckpt_loss:.4f}, MAE: {ckpt_mae:.4f}")
print(f"与原始模型一致: {np.isclose(original_loss, ckpt_loss)}")

## 4. 训练中保存 Checkpoint + EarlyStopping

在 Keras 中使用 `ModelCheckpoint` 和 `EarlyStopping` 回调，
在 PyTorch 中需要在训练循环中手动实现。

In [ ]:
class EarlyStopping:
    """早停机制
    Early stopping mechanism to stop training when validation metric stops improving.

    Args:
        patience (int): 等待多少个 epoch 无改善后停止 / Number of epochs to wait.
        delta (float): 判定为改善的最小变化 / Minimum change to qualify as improvement.
        verbose (bool): 是否打印信息 / Whether to print messages.
    """
    def __init__(self, patience=5, delta=0.0, verbose=True):
        self.patience = patience
        self.delta = delta
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_loss = float('inf')

    def __call__(self, val_loss):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.best_loss = val_loss
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f"  EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_loss = val_loss
            self.counter = 0


def train_with_checkpoint_and_earlystop(
    model, train_loader, valid_loader,
    checkpoint_path='saved_models/best_model.pth',
    epochs=50, lr=1e-3, patience=5, device=device
):
    """带 Checkpoint 和 EarlyStopping 的训练循环
    Training loop with checkpoint saving and early stopping.

    Args:
        model: PyTorch 模型 / PyTorch model.
        train_loader: 训练数据加载器 / Training DataLoader.
        valid_loader: 验证数据加载器 / Validation DataLoader.
        checkpoint_path: 最佳模型保存路径 / Path to save best model.
        epochs: 最大训练轮数 / Maximum number of epochs.
        lr: 学习率 / Learning rate.
        patience: 早停耐心值 / Early stopping patience.
        device: 计算设备 / Compute device.

    Returns:
        model, optimizer, history
    """
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    early_stopping = EarlyStopping(patience=patience, verbose=True)

    history = {'train_loss': [], 'val_loss': [], 'val_mae': []}
    best_val_loss = float('inf')

    for epoch in range(1, epochs + 1):
        # 训练 / Train
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        # 验证 / Validate
        val_loss, val_mae = evaluate(model, valid_loader, criterion, device)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_mae'].append(val_mae)

        # 保存最佳模型 Checkpoint / Save best model checkpoint
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), checkpoint_path)
            if epoch % 5 != 0 and epoch != 1:
                print(f"Epoch {epoch:3d} | val_loss: {val_loss:.4f} | val_mae: {val_mae:.4f} | ✓ 已保存最佳模型")

        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d} | train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f} | val_mae: {val_mae:.4f}")

        # 早停检查 / Early stopping check
        early_stopping(val_loss)
        if early_stopping.early_stop:
            print(f"\n早停触发! 在第 {epoch} 个 epoch 停止训练。")
            break

    return model, optimizer, history


# 训练 / Train
checkpoint_model = create_model()
checkpoint_model, opt, ckpt_history = train_with_checkpoint_and_earlystop(
    checkpoint_model, train_loader, valid_loader,
    checkpoint_path='saved_models/best_model.pth',
    epochs=50, patience=5
)

In [ ]:
# 加载最佳模型并对比 / Load best model and compare
best_model = create_model()
best_model.load_state_dict(torch.load('saved_models/best_model.pth', weights_only=True))
best_model.to(device)

# 最终模型 vs 最佳模型 / Final model vs best model
final_loss, final_mae = evaluate(checkpoint_model, test_loader, criterion, device)
best_loss, best_mae = evaluate(best_model, test_loader, criterion, device)

print("模型对比:")
print(f"最终模型 - MSE: {final_loss:.4f}, MAE: {final_mae:.4f}")
print(f"最佳模型 - MSE: {best_loss:.4f}, MAE: {best_mae:.4f}")

## 5. 保存优化器状态 — 恢复训练

当需要从某个 Checkpoint 恢复训练时，除了模型权重，还必须保存优化器的 `state_dict`。
否则优化器内部的状态（如 Adam 的动量）会丢失，导致恢复训练时性能下降。

In [ ]:
# 保存模型 + 优化器状态 / Save model + optimizer state
resume_checkpoint_path = 'saved_models/resume_checkpoint.pth'

resume_model = create_model()
resume_optimizer = optim.Adam(resume_model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

# 先训练 10 个 epoch / Train for 10 epochs first
print("=== 第一阶段训练 (10 epochs) ===")
for epoch in range(1, 11):
    train_loss = train_one_epoch(resume_model, train_loader, criterion, resume_optimizer, device)
    val_loss, val_mae = evaluate(resume_model, valid_loader, criterion, device)
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f}")

# 保存 Checkpoint（含 epoch、模型、优化器）/ Save checkpoint with epoch, model, optimizer
checkpoint = {
    'epoch': 10,
    'model_state_dict': resume_model.state_dict(),
    'optimizer_state_dict': resume_optimizer.state_dict(),
    'train_loss': train_loss,
    'val_loss': val_loss,
}
torch.save(checkpoint, resume_checkpoint_path)
print(f"\nCheckpoint 已保存到: {resume_checkpoint_path}")

In [ ]:
# 恢复训练 / Resume training
print("=== 恢复训练 (从 epoch 11 开始) ===")

# 创建新的模型和优化器实例 / Create new model and optimizer instances
resumed_model = create_model()
resumed_optimizer = optim.Adam(resumed_model.parameters(), lr=1e-3)

# 加载 Checkpoint / Load checkpoint
loaded_ckpt = torch.load(resume_checkpoint_path, weights_only=False)
resumed_model.load_state_dict(loaded_ckpt['model_state_dict'])
resumed_optimizer.load_state_dict(loaded_ckpt['optimizer_state_dict'])
start_epoch = loaded_ckpt['epoch'] + 1

print(f"从 epoch {start_epoch} 恢复训练，上次 val_loss: {loaded_ckpt['val_loss']:.4f}")

# 继续训练 10 个 epoch / Continue training for 10 more epochs
for epoch in range(start_epoch, start_epoch + 10):
    train_loss = train_one_epoch(resumed_model, train_loader, criterion, resumed_optimizer, device)
    val_loss, val_mae = evaluate(resumed_model, valid_loader, criterion, device)
    if epoch % 5 == 0 or epoch == start_epoch:
        print(f"Epoch {epoch:3d} | train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f}")

resumed_loss, resumed_mae = evaluate(resumed_model, test_loader, criterion, device)
print(f"\n恢复训练后模型 - MSE: {resumed_loss:.4f}, MAE: {resumed_mae:.4f}")

## 6. 保存自定义模型

自定义模型只需要正确实现 `__init__` 和 `forward`，
`state_dict` / `load_state_dict` 会自动处理所有 `nn.Module` 子模块的参数。

In [ ]:
# 自定义模型 / Custom model
class CustomMLP(nn.Module):
    """带有残差连接的自定义 MLP
    Custom MLP with residual connection.

    Args:
        input_dim (int): 输入维度 / Input dimension.
        hidden_dim (int): 隐藏层维度 / Hidden dimension.
        use_residual (bool): 是否使用残差连接 / Whether to use residual connection.
    """
    def __init__(self, input_dim=8, hidden_dim=30, use_residual=True):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.use_residual = use_residual

        self.hidden1 = nn.Linear(input_dim, hidden_dim)
        self.hidden2 = nn.Linear(hidden_dim, hidden_dim)
        self.output_layer = nn.Linear(hidden_dim, 1)
        self.relu = nn.ReLU()

        # 非参数层不需要特殊处理 / Non-parameterized layers need no special handling
        # use_residual 不是 nn.Parameter，不会出现在 state_dict 中
        # use_residual is not an nn.Parameter, won't appear in state_dict

    def forward(self, x):
        x = self.relu(self.hidden1(x))
        identity = x
        x = self.relu(self.hidden2(x))
        if self.use_residual:
            x = x + identity  # 残差连接 / Residual connection
        return self.output_layer(x)


# 创建并训练 / Create and train
custom_model = CustomMLP(input_dim=8, hidden_dim=30, use_residual=True).to(device)
custom_model, custom_opt, custom_hist = train_model(custom_model, train_loader, valid_loader, epochs=10)

custom_loss, custom_mae = evaluate(custom_model, test_loader, criterion, device)
print(f"\n自定义模型 - MSE: {custom_loss:.4f}, MAE: {custom_mae:.4f}")

In [ ]:
# 保存自定义模型 / Save custom model
custom_path = 'saved_models/custom_model.pth'

# 推荐方式：保存 state_dict + 构造参数 / Recommended: save state_dict + constructor args
custom_checkpoint = {
    'model_state_dict': custom_model.state_dict(),
    'model_class': custom_model.__class__.__name__,
    'input_dim': custom_model.input_dim,
    'hidden_dim': custom_model.hidden_dim,
    'use_residual': custom_model.use_residual,
}
torch.save(custom_checkpoint, custom_path)
print(f"自定义模型已保存到: {custom_path}")

# 加载自定义模型 / Load custom model
loaded_custom_ckpt = torch.load(custom_path, weights_only=False)
restored_custom = CustomMLP(
    input_dim=loaded_custom_ckpt['input_dim'],
    hidden_dim=loaded_custom_ckpt['hidden_dim'],
    use_residual=loaded_custom_ckpt['use_residual'],
).to(device)
restored_custom.load_state_dict(loaded_custom_ckpt['model_state_dict'])

rc_loss, rc_mae = evaluate(restored_custom, test_loader, criterion, device)
print(f"加载的自定义模型 - MSE: {rc_loss:.4f}, MAE: {rc_mae:.4f}")
print(f"与原始模型一致: {np.isclose(custom_loss, rc_loss)}")

## 7. ONNX 导出 — 用于部署

PyTorch 模型可以通过 ONNX 格式导出，实现在不同推理框架（TensorRT、OpenVINO、ONNX Runtime 等）中部署。
这相当于 TensorFlow 中使用 SavedModel 格式导出用于 TF Serving 的做法。

In [ ]:
# ONNX 导出 / ONNX export
onnx_path = 'saved_models/my_model.onnx'

# 创建 dummy 输入用于追踪计算图 / Create dummy input for tracing
dummy_input = torch.randn(1, 8).to(device)

model.eval()  # 必须切换到评估模式 / Must switch to eval mode

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,          # 导出模型参数 / Export model parameters
    opset_version=13,            # ONNX 算子集版本 / ONNX opset version
    do_constant_folding=True,    # 常量折叠优化 / Constant folding optimization
    input_names=['input'],       # 输入名称 / Input names
    output_names=['output'],     # 输出名称 / Output names
    dynamic_axes={               # 动态批次维度 / Dynamic batch dimension
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'},
    }
)

print(f"模型已导出为 ONNX: {onnx_path}")
print(f"文件大小: {os.path.getsize(onnx_path) / 1024:.2f} KB")

In [ ]:
# 验证 ONNX 模型 / Validate ONNX model
try:
    import onnx
    import onnxruntime as ort

    # 加载并检查 ONNX 模型 / Load and check ONNX model
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    print("✓ ONNX 模型验证通过")

    # 使用 ONNX Runtime 推理 / Inference with ONNX Runtime
    ort_session = ort.InferenceSession(onnx_path)

    # 准备输入 / Prepare input
    test_input = X_test_t[:5].numpy()
    ort_inputs = {ort_session.get_inputs()[0].name: test_input}

    # 推理 / Inference
    ort_outputs = ort_session.run(None, ort_inputs)

    # 与 PyTorch 结果对比 / Compare with PyTorch results
    model.eval()
    with torch.no_grad():
        pt_outputs = model(X_test_t[:5].to(device)).cpu().numpy()

    max_diff = np.max(np.abs(ort_outputs[0] - pt_outputs))
    print(f"\nONNX Runtime vs PyTorch 最大差异: {max_diff:.2e}")
    print("ONNX 导出验证成功!" if max_diff < 1e-5 else "⚠️ 差异较大，请检查导出")

except ImportError:
    print("请安装 onnx 和 onnxruntime 来验证 ONNX 模型:")
    print("  pip install onnx onnxruntime")

## 8. TF vs PyTorch 对照

| 功能 | TensorFlow / Keras | PyTorch |
|------|-------------------|---------|
| 保存完整模型 | `model.save(path)` (SavedModel/HDF5/.keras) | `torch.save(model, path)` (不推荐) |
| 保存权重 | `model.save_weights(path)` | `torch.save(model.state_dict(), path)` (推荐) |
| 加载完整模型 | `keras.models.load_model(path)` | `model = torch.load(path)` (不推荐) |
| 加载权重 | `model.load_weights(path)` | `model.load_state_dict(torch.load(path))` (推荐) |
| 常用文件格式 | `.h5` / `.keras` / SavedModel 目录 | `.pt` / `.pth` |
| 训练中保存 | `ModelCheckpoint` 回调 | 手动在训练循环中 `torch.save()` |
| 早停 | `EarlyStopping` 回调 | 手动实现 `EarlyStopping` 类 |
| 保存优化器状态 | 包含在完整模型中 | `optimizer.state_dict()` 单独保存 |
| 自定义对象 | `get_config` / `custom_objects` | 构造参数保存在 checkpoint 中 |
| 部署导出 | SavedModel → TF Serving | ONNX → ONNX Runtime / TensorRT / OpenVINO |
| 序列化方式 | Protocol Buffers + HDF5 | pickle (默认) 或自定义 |

### 核心差异说明

1. **PyTorch 不保存计算图**: `state_dict` 只保存参数值，模型结构由 Python 类定义决定；TF 的 SavedModel 包含完整计算图
2. **PyTorch 需要手动管理 checkpoint**: 没有 Keras 的回调机制，需要在训练循环中显式保存
3. **PyTorch 的 optimizer 需要单独保存**: 恢复训练时必须同时加载优化器状态
4. **部署路径不同**: TF 使用 TF Serving，PyTorch 通常通过 ONNX 导出后用 ONNX Runtime / TensorRT 部署

## 9. 清理保存的文件

In [ ]:
# 列出所有保存的文件 / List all saved files
print("已保存的模型文件:")
if os.path.exists('saved_models'):
    for item in sorted(os.listdir('saved_models')):
        item_path = os.path.join('saved_models', item)
        if os.path.isdir(item_path):
            print(f"  [目录] {item}")
        else:
            size = os.path.getsize(item_path) / 1024
            print(f"  [文件] {item} ({size:.2f} KB)")

# 取消注释以下代码来清理文件 / Uncomment to clean up files
# shutil.rmtree('saved_models')
# print("\n已清理所有保存的文件")

## 小结

### PyTorch 保存格式选择

| 方式 | 文件 | 优点 | 缺点 |
|------|------|------|------|
| `state_dict` | `.pt` / `.pth` | 推荐、灵活、安全 | 需要代码中定义模型类 |
| 完整模型 | `.pt` | 简单快速 | pickle 依赖、跨环境不可靠 |
| Checkpoint | `.pth` | 可保存 epoch、优化器等 | 需要手动管理 |
| ONNX | `.onnx` | 跨框架部署 | 部分算子不支持 |

### 最佳实践

1. **始终保存 `state_dict` 而非完整模型**: 更安全、更灵活
2. **训练中使用 Checkpoint + EarlyStopping**: 自动保存最佳模型，避免过拟合
3. **恢复训练时保存优化器状态**: 保持优化器动量等信息
4. **将构造参数保存在 checkpoint 中**: 便于重建模型
5. **生产部署使用 ONNX 导出**: 高性能跨平台推理
6. **使用 `weights_only=True` 加载**: 防止 pickle 反序列化安全风险

## 练习

### 练习 1：实现定期 Checkpoint

修改训练循环，每 5 个 epoch 保存一次 checkpoint（文件名包含 epoch 编号），
如 `checkpoint_epoch_5.pth`、`checkpoint_epoch_10.pth` 等。
同时保留最佳模型的保存逻辑。

```python
# TODO: 实现定期 checkpoint 保存
# 提示：在训练循环中判断 epoch % 5 == 0
```

### 练习 2：迁移学习权重加载

假设你已经训练好一个 `MLP(input_dim=8, hidden_dim=30)` 模型，
现在要创建一个 `MLP(input_dim=8, hidden_dim=50)` 的新模型，
并尽可能加载旧模型的权重（隐藏层维度不同时跳过该层）。

```python
# TODO: 实现部分权重加载
# 提示：使用 model.load_state_dict(state_dict, strict=False)
# 或者手动过滤 state_dict 的键
```

### 练习 3：完整的可恢复训练框架

编写一个函数 `train_resumable(model, train_loader, valid_loader, ckpt_dir, ...)`，
支持以下功能：
- 如果 `ckpt_dir` 中存在最新 checkpoint，则自动恢复训练
- 每个 epoch 结束后保存 checkpoint
- 只保留最近 3 个 checkpoint（自动删除旧的）
- 保存最佳模型到单独文件

```python
# TODO: 实现完整的可恢复训练框架
# 提示：使用 sorted + os.listdir 找到最新 checkpoint
# 使用 glob 或 os.listdir 过滤旧 checkpoint
```